In [ ]:
# Install required libraries with error handling
!echo "Installing dependencies..." && !pip install tensorflow-gpu numpy pandas scikit-learn sentence-transformers transformers torch -q
!echo "Installing chromadb..." && !pip install chromadb --no-cache-dir -q
print("All dependencies installed successfully.")

Installing dependencies...
/bin/bash: line 1: !pip: command not found
Installing chromadb...
/bin/bash: line 1: !pip: command not found
All dependencies installed successfully.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
import tensorflow as tf
from sklearn.model_selection import train_test_split

# Unzip the emotion_db if not already extracted
!unzip -o emotion_db.zip -d ./emotion_db 2>/dev/null || echo "emotion_db already extracted or zip not found."

# Load the dataset
file_path = '/content/NRC-VAD-Lexicon-v2.1.txt'
data = pd.read_csv(file_path, sep='\t', header=None, names=['Word', 'Valence', 'Arousal', 'Dominance'], skiprows=1)
data = data.dropna()
data[['Valence', 'Arousal', 'Dominance']] = data[['Valence', 'Arousal', 'Dominance']].apply(pd.to_numeric, errors='coerce')
data = data.dropna()

# Normalize VAD scores to [-1, 1]
scaler = MinMaxScaler(feature_range=(-1, 1))
vad_scores = scaler.fit_transform(data[['Valence', 'Arousal', 'Dominance']])

# Cluster VAD scores into 40 emotion categories
kmeans = KMeans(n_clusters=40, random_state=42, n_init=10)
emotion_clusters = kmeans.fit_predict(vad_scores)

# Reshape VAD scores for CNN input (samples, 3, 1)
X_full = vad_scores.reshape((vad_scores.shape[0], vad_scores.shape[1], 1))
y_full = tf.keras.utils.to_categorical(emotion_clusters, 40)

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X_full, y_full, test_size=0.2, random_state=42)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

Archive:  emotion_db.zip
   creating: ./emotion_db/emotion_db/
  inflating: ./emotion_db/emotion_db/chroma.sqlite3  
   creating: ./emotion_db/emotion_db/ca9dd079-d065-4a8e-b8a5-300bd196429f/
 extracting: ./emotion_db/emotion_db/ca9dd079-d065-4a8e-b8a5-300bd196429f/link_lists.bin  
  inflating: ./emotion_db/emotion_db/ca9dd079-d065-4a8e-b8a5-300bd196429f/header.bin  
  inflating: ./emotion_db/emotion_db/ca9dd079-d065-4a8e-b8a5-300bd196429f/length.bin  
  inflating: ./emotion_db/emotion_db/ca9dd079-d065-4a8e-b8a5-300bd196429f/data_level0.bin  
Train shape: (43840, 3, 1), Test shape: (10960, 3, 1)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

# Build the CNN model with 40 outputs
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"Using GPU: {gpus[0]}")
    tf.config.experimental.set_memory_growth(gpus[0], True)
else:
    print("No GPU found, using CPU.")

model = Sequential([
    Conv1D(filters=64, kernel_size=2, activation='relu', input_shape=(3, 1), padding='same'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    Conv1D(filters=128, kernel_size=1, activation='relu', padding='same'),
    BatchNormalization(),
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(40, activation='softmax')  # 40 clusters
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model with early stopping, optimized for T4 GPU
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=50, batch_size=128,  # Optimized for T4
                    validation_data=(X_test, y_test), callbacks=[early_stopping], verbose=1)

# Save the trained weights
model.save_weights('improved_cnn_weights_40clusters.weights.h5')
print("Model trained and weights saved as 'improved_cnn_weights_40clusters.weights.h5'.")

Using GPU: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
343/343 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - accuracy: 0.6030 - loss: 1.2786 - val_accuracy: 0.1920 - val_loss: 3.3807
Epoch 2/50
343/343 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8125 - loss: 0.4733 - val_accuracy: 0.8261 - val_loss: 0.4349
Epoch 3/50
343/343 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8430 - loss: 0.3927 - val_accuracy: 0.8708 - val_loss: 0.3275
Epoch 4/50
343/343 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8502 - loss: 0.3674 - val_accuracy: 0.8804 - val_loss: 0.2854
Epoch 5/50
343/343 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8606 - loss: 0.3463 - val_accuracy: 0.8767 - val_loss: 0.2923
Epoch 6/50
343/343 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8676 - loss: 0.3278 - val_accuracy: 0.9020 - val_loss: 0.2371
Epoch 7/50
343/343 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8780 - loss: 0.2997 - val_accuracy: 0.8802 - val_loss: 0.2817
Epoch 8/50
343/343 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8804 - loss: 0.2994 - val_accuracy: 

In [ ]:
# Evaluate on test set
loss, accuracy = model.evaluate(X_test, y_test, batch_size=128, verbose=1)
print(f"Test Set - Loss: {loss:.4f}, Accuracy: {accuracy:.4f}")

# Predict and compute metrics
y_pred = model.predict(X_test, batch_size=128, verbose=1)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

from sklearn.metrics import precision_score, recall_score, f1_score
precision = precision_score(y_true_classes, y_pred_classes, average='weighted')
recall = recall_score(y_true_classes, y_pred_classes, average='weighted')
f1 = f1_score(y_true_classes, y_pred_classes, average='weighted')

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9177 - loss: 0.1953
Test Set - Loss: 0.1934, Accuracy: 0.9189
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
Precision: 0.9226
Recall: 0.9189
F1-Score: 0.9189


In [ ]:
# Install chromadb
!echo "Installing chromadb..."
!pip install chromadb --no-cache-dir -q
print("ChromaDB installation completed.")

# Import libraries
import chromadb
from sentence_transformers import SentenceTransformer

# Initialize ChromaDB client
try:
    chroma_client = chromadb.PersistentClient(path="./emotion_db/emotion_db")
    collection = chroma_client.get_or_create_collection(name="emotions")
    embedder = SentenceTransformer('all-MiniLM-L6-v2')
    print("ChromaDB and embedder initialized successfully.")
except Exception as e:
    print(f"Error initializing ChromaDB: {e}. Ensure emotion_db is extracted.")

# Verify database contents
try:
    count = collection.count()
    metadata = collection.get(include=["metadatas"])["metadatas"]
    emotions = [item["emotion"] for item in metadata] if metadata else []
    print(f"Knowledge Base - Number of Entries: {count}")
    print(f"Stored Emotions: {emotions if emotions else 'None'}")
except Exception as e:
    print(f"Error accessing database: {e}")

Installing chromadb...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 296.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 363.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 321.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.1/103.1 kB 310.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 225.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 323.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 324.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.0/120.0 kB 379.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.6/201.6 kB 352.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ChromaDB and embedder initialized successfully.
Knowledge Base - Number of Entries: 40
Stored Emotions: ['Happy', 'Excited', 'Confident', 'Enthusiastic', 'Proud', 'Thrilled', 'Delighted', 'Overjoyed', 'Giddy', 'Exhilarated', 'Calm', 'Content', 'Satisfied', 'Secure', 'At Ease', 'Affectionate', 'Fond', 'Warm', 'Tender', 'Sentimental', 'Angry', 'Frustrated', 'Annoyed', 'Disgusted', 'Contemptuous', 'Scared', 'Nervous', 'Worried', 'Frightened', 'Alarmed', 'Disdainful', 'Scornful', 'Condescending', 'Dismissive', 'Supercilious', 'Sad', 'Depressed', 'Melancholy', 'Despondent', 'Hopeless']


In [ ]:
def predict_closest_word(vad_scores):
    if len(vad_scores) != 3:
        raise ValueError("VAD scores must contain exactly 3 values (valence, arousal, dominance).")
    vad_array = np.array(vad_scores).reshape(1, 3, 1).astype(np.float32)
    scaler = tf.keras.layers.Normalization(axis=-1)
    scaler.adapt(np.array([[[-1], [-1], [-1]], [[1], [1], [1]]]).astype(np.float32))
    vad_normalized = scaler(vad_array).numpy()
    prediction = model.predict(vad_normalized, verbose=0)
    cluster = np.argmax(prediction, axis=1)[0]
    return cluster

def detect_emotion_from_rag(closest_word):
    try:
        query_embedding = embedder.encode([closest_word], convert_to_tensor=False).tolist()
        results = collection.query(query_embeddings=query_embedding, n_results=1)
        if results['metadatas'][0]:
            matched_emotion = results['metadatas'][0][0]['emotion']
            description = results['documents'][0][0]
            distance = results['distances'][0][0]
            return matched_emotion, description, distance
        return "None", "No match found", float('inf')
    except Exception as e:
        print(f"RAG query error: {e}")
        return "None", "No match found", float('inf')

def generate_musicgen_prompt(vad_scores, detected_emotion, description):
    valence, arousal, dominance = vad_scores
    prompt = f"""
    Compose therapeutic music for '{detected_emotion}'.
    VAD: Valence={valence} (-1=very unpleasant, 1=very pleasant), Arousal={arousal} (-1=very calm, 1=very energetic), Dominance={dominance} (-1=very submissive, 1=very dominant).
    Description: {description}
    Use suitable tempo, key, and instruments for emotional therapy.
    """
    return prompt.strip()

In [ ]:
from google.colab import output

def emotion_to_musicgen_pipeline(vad_scores):
    if not all(-1 <= x <= 1 for x in vad_scores):
        return "Error: VAD scores must be between -1 and 1."
    if len(vad_scores) != 3:
        return "Error: VAD scores must contain exactly 3 values (valence, arousal, dominance)."

    # Predict closest word cluster
    cluster = predict_closest_word(vad_scores)
    emotions_list = ['Happy', 'Excited', 'Confident', 'Enthusiastic', 'Proud', 'Thrilled', 'Delighted', 'Overjoyed',
                     'Giddy', 'Exhilarated', 'Calm', 'Content', 'Satisfied', 'Secure', 'At Ease', 'Affectionate',
                     'Fond', 'Warm', 'Tender', 'Sentimental', 'Angry', 'Frustrated', 'Annoyed', 'Disgusted',
                     'Contemptuous', 'Scared', 'Nervous', 'Worried', 'Frightened', 'Alarmed', 'Disdainful',
                     'Scornful', 'Condescending', 'Dismissive', 'Supercilious', 'Sad', 'Depressed', 'Melancholy',
                     'Despondent', 'Hopeless']
    word_mapping = emotions_list[cluster] if 0 <= cluster < len(emotions_list) else "Unknown"
    print(f"Predicted Cluster: {word_mapping}")

    # Detect emotion using RAG
    detected_emotion, description, distance = detect_emotion_from_rag(word_mapping)
    print(f"Detected Emotion from Knowledge Base: {detected_emotion}, Similarity Distance: {distance:.4f}")

    # Generate MusicGen prompt
    musicgen_prompt = generate_musicgen_prompt(vad_scores, detected_emotion, description)
    print(f"\nMusicGen Prompt:\n{musicgen_prompt}")

    return musicgen_prompt

# Get user input for VAD scores
valence = float(input("Enter Valence score (-1 to 1): "))
arousal = float(input("Enter Arousal score (-1 to 1): "))
dominance = float(input("Enter Dominance score (-1 to 1): "))
vad_scores = [valence, arousal, dominance]

# Clear output and run pipeline
output.clear()
result = emotion_to_musicgen_pipeline(vad_scores)
print(f"\nFinal Result: {result}")

Predicted Cluster: Condescending
Detected Emotion from Knowledge Base: Condescending, Similarity Distance: 1.6338

MusicGen Prompt:
Compose therapeutic music for 'Condescending'.
    VAD: Valence=0.512 (-1=very unpleasant, 1=very pleasant), Arousal=0.0 (-1=very calm, 1=very energetic), Dominance=0.0 (-1=very submissive, 1=very dominant).
    Description: Enhanced Therapeutic Approach: Transform patronizing attitudes through exposure to genuinely superior musical artistry that promotes humility while engaging intellectual faculties. Use complex classical forms (fugue, sonata form) with sophisticated developmental techniques and advanced harmonic language. Employ varied tempos that demonstrate musical sophistication through appropriate style and character, using complex meters and phrase structures that require advanced musical understanding. Choose instruments associated with classical tradition and refined musical culture: piano with sophisticated pedaling techniques, string quartet wi

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Load the dataset
file_path = '/content/NRC-VAD-Lexicon-v2.1.txt'
data = pd.read_csv(file_path, sep='\t', header=None, names=['Word', 'Valence', 'Arousal', 'Dominance'], skiprows=1)
data = data.dropna()
data[['Valence', 'Arousal', 'Dominance']] = data[['Valence', 'Arousal', 'Dominance']].apply(pd.to_numeric, errors='coerce')
data = data.dropna()

# Normalize VAD scores to [-1, 1]
scaler = MinMaxScaler(feature_range=(-1, 1))
vad_scores = scaler.fit_transform(data[['Valence', 'Arousal', 'Dominance']])

# Cluster VAD scores into 40 emotion categories
kmeans = KMeans(n_clusters=40, random_state=42, n_init=10)
emotion_clusters = kmeans.fit_predict(vad_scores)

# Reshape VAD scores for CNN input (samples, 3, 1)
X_full = vad_scores.reshape((vad_scores.shape[0], vad_scores.shape[1], 1))
y_full = tf.keras.utils.to_categorical(emotion_clusters, 40)

# Build the CNN model with 40 outputs
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"Using GPU: {gpus[0]}")
    tf.config.experimental.set_memory_growth(gpus[0], True)
else:
    print("No GPU found, using CPU.")

model = Sequential([
    Conv1D(filters=64, kernel_size=2, activation='relu', input_shape=(3, 1), padding='same'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    Conv1D(filters=128, kernel_size=1, activation='relu', padding='same'),
    BatchNormalization(),
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(40, activation='softmax')  # 40 clusters
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Load pre-trained weights (ensure file exists)
try:
    model.load_weights('/content/improved_cnn_weights_40clusters.weights.h5')
    print("Weights loaded successfully.")
except Exception as e:
    print(f"Error loading weights: {e}. Please ensure 'improved_cnn_weights_40clusters.weights.h5' is uploaded.")
    exit()

# Evaluate the model on the entire dataset
y_pred = model.predict(X_full, batch_size=128, verbose=1)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_full, axis=1)

# Compute metrics
accuracy = accuracy_score(y_true_classes, y_pred_classes)
precision = precision_score(y_true_classes, y_pred_classes, average='weighted')
recall = recall_score(y_true_classes, y_pred_classes, average='weighted')
f1 = f1_score(y_true_classes, y_pred_classes, average='weighted')

# Print results
print(f"Evaluation on Entire Dataset - Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

Using GPU: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
Weights loaded successfully.


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 26 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


429/429 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Evaluation on Entire Dataset - Accuracy: 0.9220
Precision: 0.9253
Recall: 0.9220
F1-Score: 0.9220
